# Loopy Colab Research Notebook

Use this notebook for both the legacy v2 baseline path and the current clean-data benchmark path.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

repo_dir = Path('/content/googlereview')
zip_path = Path('/content/googlereview.zip')
extract_dir = Path('/content/googlereview-main')

if repo_dir.exists():
    shutil.rmtree(repo_dir)
if zip_path.exists():
    zip_path.unlink()
if extract_dir.exists():
    shutil.rmtree(extract_dir)

subprocess.run(['wget', '-q', '-O', str(zip_path), 'https://codeload.github.com/darwesh88/googlereview/zip/refs/heads/main'], check=True)
subprocess.run(['unzip', '-q', str(zip_path), '-d', '/content'], check=True)
extract_dir.rename(repo_dir)
os.chdir(repo_dir)
print('Ready in', os.getcwd())
print('Top-level files:')
print('\n'.join(sorted(os.listdir('.'))[:20]))


In [ ]:
import os
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=True)
req_path = os.path.join('loopy', 'requirements.txt')
if os.path.exists(req_path):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', req_path], check=True)
else:
    print('No loopy/requirements.txt found, continuing')


In [ ]:
import os
import torch

print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))

dataset_path = 'loopy/data/real/twitter_support_5k.txt'
print('dataset exists', os.path.exists(dataset_path), dataset_path)


## Clean Benchmark Track (TinyStories)

Run this section top to bottom to compare raw vs best `v3` vs best `v4.2` on a cleaner dataset before doing more noisy-support sweeps.


In [ ]:
!python -m loopy.prepare_hf_corpus \
  --dataset tinystories \
  --output loopy/data/real/tinystories_5k.txt \
  --max-samples 5000 \
  --min-chars 40 \
  --max-chars 600 \
  --min-tokens 6 \
  --dedupe


In [ ]:
!cat loopy/data/real/tinystories_5k.report.json


### 1. Raw Patch Prior Baseline on TinyStories


In [ ]:
!python -m loopy.train_patch_prior_v2 \
  --mode raw \
  --data-path loopy/data/real/tinystories_5k.txt \
  --output-dir loopy/runs/auto_clean/raw_p2_20ep_clean/prior \
  --batch-size 16 \
  --epochs 20 \
  --learning-rate 0.001 \
  --weight-decay 0.01 \
  --hidden-size 192 \
  --num-layers 2 \
  --dropout 0.1 \
  --max-seq-len 128 \
  --patch-size 2 \
  --byte-embed-dim 16 \
  --seed 7


In [ ]:
!cat loopy/runs/auto_clean/raw_p2_20ep_clean/prior/best_metrics.json


### 2. Best Downstream `v3` Reference on TinyStories


In [ ]:
!python -m loopy.train_symbolic_codec_v3 \
  --data-path loopy/data/real/tinystories_5k.txt \
  --output-dir loopy/runs/auto_clean/v3_5bpb_clean/codec \
  --epochs 20 \
  --batch-size 8 \
  --max-seq-len 128 \
  --patch-size 2 \
  --embed-dim 128 \
  --latent-dim 128 \
  --encoder-layers 2 \
  --decoder-layers 2 \
  --num-heads 4 \
  --dropout 0.0 \
  --weight-decay 0.0 \
  --num-codebooks 2 \
  --sub-codebook-size 32 \
  --assignment-temp 0.5 \
  --commitment-weight 0.05 \
  --codebook-weight 0.25 \
  --usage-weight 0.05 \
  --predictive-weight 0.0


In [ ]:
!cat loopy/runs/auto_clean/v3_5bpb_clean/codec/best_metrics.json
print()
!cat loopy/runs/auto_clean/v3_5bpb_clean/codec/sample_reconstruction.txt


In [ ]:
!python -m loopy.train_patch_prior_v3 \
  --data-path loopy/data/real/tinystories_5k.txt \
  --codec-run-dir loopy/runs/auto_clean/v3_5bpb_clean/codec \
  --output-dir loopy/runs/auto_clean/v3_5bpb_clean/prior \
  --epochs 20 \
  --batch-size 16 \
  --hidden-size 128 \
  --num-layers 2 \
  --dropout 0.1 \
  --learning-rate 0.001 \
  --weight-decay 0.01 \
  --max-seq-len 128 \
  --patch-size 2 \
  --group-embed-dim 16 \
  --batch-encode-size 32


In [ ]:
!cat loopy/runs/auto_clean/v3_5bpb_clean/prior/best_metrics.json


### 3. Best Balanced `v4.2` Reference on TinyStories


In [ ]:
!python -m loopy.train_symbolic_codec_v4 \
  --data-path loopy/data/real/tinystories_5k.txt \
  --output-dir loopy/runs/auto_clean/v42_6bpb_clean/codec \
  --epochs 20 \
  --batch-size 8 \
  --max-seq-len 128 \
  --patch-size 2 \
  --embed-dim 128 \
  --latent-dim 128 \
  --encoder-layers 2 \
  --decoder-layers 2 \
  --pre-context-layers 1 \
  --post-context-layers 1 \
  --num-heads 4 \
  --dropout 0.0 \
  --weight-decay 0.0 \
  --num-codebooks 2 \
  --sub-codebook-size 64 \
  --assignment-temp 0.5 \
  --commitment-weight 0.05 \
  --codebook-weight 0.25 \
  --usage-weight 0.05 \
  --predictive-weight 0.0 \
  --predictive-mask-prob 0.15 \
  --use-residual-detail \
  --residual-usage-weight 0.005 \
  --residual-gate-bias -2.0


In [ ]:
!cat loopy/runs/auto_clean/v42_6bpb_clean/codec/best_metrics.json
print()
!cat loopy/runs/auto_clean/v42_6bpb_clean/codec/sample_reconstruction.txt


In [ ]:
!python -m loopy.train_patch_prior_v4 \
  --data-path loopy/data/real/tinystories_5k.txt \
  --codec-run-dir loopy/runs/auto_clean/v42_6bpb_clean/codec \
  --output-dir loopy/runs/auto_clean/v42_6bpb_clean/prior \
  --epochs 20 \
  --batch-size 16 \
  --hidden-size 128 \
  --num-layers 2 \
  --dropout 0.1 \
  --learning-rate 0.001 \
  --weight-decay 0.01 \
  --max-seq-len 128 \
  --patch-size 2 \
  --group-embed-dim 16 \
  --batch-encode-size 32


In [ ]:
!cat loopy/runs/auto_clean/v42_6bpb_clean/prior/best_metrics.json


### 4. Best Masked-Predictive `v4.2` Reference on TinyStories


In [ ]:
!python -m loopy.train_symbolic_codec_v4 \
  --data-path loopy/data/real/tinystories_5k.txt \
  --output-dir loopy/runs/auto_clean/v42_6bpb_maskpred_clean/codec \
  --epochs 20 \
  --batch-size 8 \
  --max-seq-len 128 \
  --patch-size 2 \
  --embed-dim 128 \
  --latent-dim 128 \
  --encoder-layers 2 \
  --decoder-layers 2 \
  --pre-context-layers 1 \
  --post-context-layers 1 \
  --num-heads 4 \
  --dropout 0.0 \
  --weight-decay 0.0 \
  --num-codebooks 2 \
  --sub-codebook-size 64 \
  --assignment-temp 0.5 \
  --commitment-weight 0.05 \
  --codebook-weight 0.25 \
  --usage-weight 0.05 \
  --predictive-weight 0.01 \
  --predictive-mask-prob 0.15 \
  --use-residual-detail \
  --residual-usage-weight 0.005 \
  --residual-gate-bias -2.0


In [ ]:
!cat loopy/runs/auto_clean/v42_6bpb_maskpred_clean/codec/best_metrics.json
print()
!cat loopy/runs/auto_clean/v42_6bpb_maskpred_clean/codec/sample_reconstruction.txt


In [ ]:
!python -m loopy.train_patch_prior_v4 \
  --data-path loopy/data/real/tinystories_5k.txt \
  --codec-run-dir loopy/runs/auto_clean/v42_6bpb_maskpred_clean/codec \
  --output-dir loopy/runs/auto_clean/v42_6bpb_maskpred_clean/prior \
  --epochs 20 \
  --batch-size 16 \
  --hidden-size 128 \
  --num-layers 2 \
  --dropout 0.1 \
  --learning-rate 0.001 \
  --weight-decay 0.01 \
  --max-seq-len 128 \
  --patch-size 2 \
  --group-embed-dim 16 \
  --batch-encode-size 32


In [ ]:
!cat loopy/runs/auto_clean/v42_6bpb_maskpred_clean/prior/best_metrics.json


### 5. Prior-Aware `v5` Benchmark on TinyStories


In [ ]:
!python -m loopy.train_symbolic_codec_v5 \
  --data-path loopy/data/real/tinystories_5k.txt \
  --output-dir loopy/runs/auto_clean/v5_6bpb_clean/codec \
  --epochs 20 \
  --batch-size 8 \
  --max-seq-len 128 \
  --patch-size 2 \
  --embed-dim 128 \
  --latent-dim 128 \
  --encoder-layers 2 \
  --decoder-layers 2 \
  --pre-context-layers 1 \
  --post-context-layers 1 \
  --num-heads 4 \
  --dropout 0.0 \
  --weight-decay 0.0 \
  --num-codebooks 2 \
  --sub-codebook-size 64 \
  --assignment-temp 0.5 \
  --commitment-weight 0.05 \
  --codebook-weight 0.25 \
  --usage-weight 0.05 \
  --use-residual-detail \
  --residual-usage-weight 0.005 \
  --residual-gate-bias -2.0 \
  --prior-weight 0.05 \
  --prior-hidden-size 128 \
  --prior-num-layers 2 \
  --prior-dropout 0.1


In [ ]:
!cat loopy/runs/auto_clean/v5_6bpb_clean/codec/best_metrics.json
print()
!cat loopy/runs/auto_clean/v5_6bpb_clean/codec/sample_reconstruction.txt


In [ ]:
!python -m loopy.train_patch_prior_v5 \
  --data-path loopy/data/real/tinystories_5k.txt \
  --codec-run-dir loopy/runs/auto_clean/v5_6bpb_clean/codec \
  --output-dir loopy/runs/auto_clean/v5_6bpb_clean/prior \
  --epochs 20 \
  --batch-size 16 \
  --hidden-size 128 \
  --num-layers 2 \
  --dropout 0.1 \
  --learning-rate 0.001 \
  --weight-decay 0.01 \
  --max-seq-len 128 \
  --patch-size 2 \
  --group-embed-dim 16 \
  --batch-encode-size 32


In [ ]:
!cat loopy/runs/auto_clean/v5_6bpb_clean/prior/best_metrics.json


### 6. Persist Results Before the Session Ends


In [ ]:
!zip -r /content/loopy_auto_clean_results.zip loopy/runs/auto_clean


## Baseline run (`rate_weight=0.0`)


In [ ]:
!python -m loopy.train_binary_codec_v2 \
  --data-path loopy/data/real/twitter_support_5k.txt \
  --output-dir loopy/runs/v2_colab_baseline \
  --epochs 20 \
  --batch-size 16 \
  --max-seq-len 128 \
  --patch-size 4 \
  --embed-dim 128 \
  --latent-dim 128 \
  --encoder-layers 2 \
  --decoder-layers 2 \
  --num-heads 4 \
  --dropout 0.0 \
  --weight-decay 0.0 \
  --bit-groups 6,6,6,6 \
  --rate-weight 0.0 \
  --balance-weight 0.001 \
  --align-weight 0.05


In [ ]:
!cat loopy/runs/v2_colab_baseline/best_metrics.json
print()
!cat loopy/runs/v2_colab_baseline/sample_reconstruction.txt


In [ ]:
!python -m loopy.measure_bitstream_v2 \
  --run-dir loopy/runs/v2_colab_baseline \
  --data-path loopy/data/real/twitter_support_5k.txt \
  --output loopy/runs/v2_colab_baseline/bitstream_summary.json
!cat loopy/runs/v2_colab_baseline/bitstream_summary.json


## Rate-aware comparison (`rate_weight=0.001`)

Run this after the baseline so the two outputs stay side by side.


In [ ]:
!python -m loopy.train_binary_codec_v2 \
  --data-path loopy/data/real/twitter_support_5k.txt \
  --output-dir loopy/runs/v2_colab_rate_small \
  --epochs 20 \
  --batch-size 16 \
  --max-seq-len 128 \
  --patch-size 4 \
  --embed-dim 128 \
  --latent-dim 128 \
  --encoder-layers 2 \
  --decoder-layers 2 \
  --num-heads 4 \
  --dropout 0.0 \
  --weight-decay 0.0 \
  --bit-groups 6,6,6,6 \
  --rate-weight 0.001 \
  --balance-weight 0.001 \
  --align-weight 0.05


In [ ]:
!cat loopy/runs/v2_colab_rate_small/best_metrics.json
print()
!cat loopy/runs/v2_colab_rate_small/sample_reconstruction.txt


In [ ]:
!python -m loopy.measure_bitstream_v2 \
  --run-dir loopy/runs/v2_colab_rate_small \
  --data-path loopy/data/real/twitter_support_5k.txt \
  --output loopy/runs/v2_colab_rate_small/bitstream_summary.json
!cat loopy/runs/v2_colab_rate_small/bitstream_summary.json
